In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/oluwafemiolasupo/project1-supply-chain-demand-csv/project1_supply_chain_demand.csv


## 1. Dataset Inspection: checked the shape and the general outlook of the dataset

In [2]:
DATA_PATH = "/kaggle/input/datasets/oluwafemiolasupo/project1-supply-chain-demand-csv/project1_supply_chain_demand.csv"

df = pd.read_csv(DATA_PATH, parse_dates=['date'])

print("Shape:", df.shape)
print("\nColumns:", list(df.columns))
print("\nDtypes:\n", df.dtypes)
print("\nHead:\n", df.head())
print("\nTail:\n", df.tail())

Shape: (4551, 7)

Columns: ['date', 'sku_id', 'category', 'units_sold', 'units_received', 'closing_stock', 'lead_time_days']

Dtypes:
 date              datetime64[ns]
sku_id                    object
category                  object
units_sold               float64
units_received             int64
closing_stock            float64
lead_time_days             int64
dtype: object

Head:
         date    sku_id category  units_sold  units_received  closing_stock  \
0 2026-01-01  SKU-1000   Snacks       110.0               0          403.0   
1 2026-01-02  SKU-1000   Snacks       278.0               0          125.0   
2 2026-01-03  SKU-1000   Snacks       121.0             903          907.0   
3 2026-01-04  SKU-1000   Snacks        75.0               0          832.0   
4 2026-01-05  SKU-1000   Snacks        71.0               0          761.0   

   lead_time_days  
0               7  
1               7  
2               7  
3               7  
4               7  

Tail:
            date

## 2. Removed nulls, duplicates, ambiguous labels, and flagged the three new SKUs

In [3]:
# Drop exact duplicate (date, sku_id) rows, keep first occurrence
before = df.shape[0]
df_clean = df.drop_duplicates(subset=['date', 'sku_id'], keep='first').copy()
print(f"Dropped {before - df_clean.shape[0]} duplicate rows -> new shape: {df_clean.shape}")

# Normalize category labels
df_clean['category'] = df_clean['category'].str.strip().str.title()
print("\nCategory value counts after normalization:")
print(df_clean['category'].value_counts())

# Missing values after cleaning
print("\nMissing values:")
print(df_clean.isna().sum())

# Flag new (cold-start) SKUs
new_skus = ['SKU-2000', 'SKU-2001', 'SKU-2002']
df_clean['is_new_sku'] = df_clean['sku_id'].isin(new_skus)
print("\nRecords per SKU (new vs established), just to sanity check counts:")
print(df_clean.groupby('is_new_sku').size())

Dropped 15 duplicate rows -> new shape: (4536, 7)

Category value counts after normalization:
category
Beverages    1272
Snacks       1092
Sugar         900
Pasta         720
Flour         552
Name: count, dtype: int64

Missing values:
date               0
sku_id             0
category           0
units_sold        90
units_received     0
closing_stock     45
lead_time_days     0
dtype: int64

Records per SKU (new vs established), just to sanity check counts:
is_new_sku
False    4500
True       36
dtype: int64


## 3. Basic stockout calculation to differentiate missing stock from actual stockout

In [4]:
# Correct stockout calculation: only count actual closing_stock == 0, exclude missing (NaN) entirely
stockout_mask = df_clean['closing_stock'] == 0
n_stockouts = stockout_mask.sum()
n_known_stock = df_clean['closing_stock'].notna().sum()

print(f"Stockout SKU-days (closing_stock == 0, missing excluded): {n_stockouts}")
print(f"Of {n_known_stock} observations with known stock: {round(n_stockouts/n_known_stock*100, 2)}%")

print("\nStockouts by category:")
print(df_clean[stockout_mask].groupby('category').size())

print("\nStockouts by month:")
print(df_clean[stockout_mask].groupby(df_clean.loc[stockout_mask, 'date'].dt.to_period('M')).size())

# Show what happens if you (incorrectly) treat missing closing_stock as a stockout —
# this reproduces the ORIGINAL (wrong) EDA numbers, to document the discrepancy
wrong_mask = (df_clean['closing_stock'] == 0) | (df_clean['closing_stock'].isna())
print(f"\n[For comparison only] If missing values were wrongly treated as stockouts: {wrong_mask.sum()} "
      f"({round(wrong_mask.sum()/df_clean.shape[0]*100,2)}%)")

Stockout SKU-days (closing_stock == 0, missing excluded): 275
Of 4491 observations with known stock: 6.12%

Stockouts by category:
category
Beverages    77
Flour        15
Pasta        16
Snacks       98
Sugar        69
dtype: int64

Stockouts by month:
date
2026-01    48
2026-02    19
2026-03    41
2026-04    44
2026-05    57
2026-06    66
Freq: M, dtype: int64

[For comparison only] If missing values were wrongly treated as stockouts: 320 (7.05%)


In [5]:
# Inventory-flow check: closing_stock[t] ?= closing_stock[t-1] + units_received[t] - units_sold[t]
df_sorted = df_clean.sort_values(['sku_id', 'date']).reset_index(drop=True)
df_sorted['prev_closing'] = df_sorted.groupby('sku_id')['closing_stock'].shift(1)
df_sorted['expected_closing'] = df_sorted['prev_closing'] + df_sorted['units_received'] - df_sorted['units_sold']
df_sorted['flow_diff'] = df_sorted['closing_stock'] - df_sorted['expected_closing']

comparable = df_sorted.dropna(subset=['prev_closing', 'units_received', 'units_sold', 'closing_stock'])
exact_match = (comparable['flow_diff'].abs() < 0.5).sum()
mismatch = (comparable['flow_diff'].abs() >= 0.5).sum()

print(f"Comparable records: {len(comparable)}")
print(f"Exact matches: {exact_match}")
print(f"Mismatches: {mismatch} ({round(mismatch/len(comparable)*100,2)}%)")
print("\nMismatch magnitude stats:")
print(comparable.loc[comparable['flow_diff'].abs()>=0.5, 'flow_diff'].describe())

# New SKU pattern check
print("\n--- New SKU behavior ---")
for sku in ['SKU-2000', 'SKU-2001', 'SKU-2002']:
    sub = df_clean[df_clean['sku_id']==sku].sort_values('date')
    print(f"\n{sku}")
    print(" closing_stock:", sub['closing_stock'].tolist())
    print(" units_received:", sub['units_received'].tolist())
    print(" lead_time_days:", sub['lead_time_days'].tolist(), "-> unique values:", sub['lead_time_days'].nunique())

Comparable records: 4330
Exact matches: 4061
Mismatches: 269 (6.21%)

Mismatch magnitude stats:
count    269.000000
mean     207.665428
std      116.770822
min        1.000000
25%      125.000000
50%      207.000000
75%      277.000000
max      640.000000
Name: flow_diff, dtype: float64

--- New SKU behavior ---

SKU-2000
 closing_stock: [437.0, 129.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
 units_received: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
 lead_time_days: [3, 14, 5, 14, 7, 3, 7, 14, 7, 7, 7, 10] -> unique values: 5

SKU-2001
 closing_stock: [729.0, 514.0, 161.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
 units_received: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
 lead_time_days: [3, 10, 10, 10, 3, 10, 14, 5, 5, 5, 3, 14] -> unique values: 4

SKU-2002
 closing_stock: [872.0, 536.0, 280.0, 80.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
 units_received: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
 lead_time_days: [7, 5, 7, 14, 5, 5, 3, 3, 14, 10, 14, 7] -> unique values: 5


## 4. Worked only with established SKUs for baseline forecasting; new SKUs get a separate cold-start path

In [6]:
# Work only with established SKUs for baseline forecasting; new SKUs get a separate cold-start path
est = df_clean[~df_clean['is_new_sku']].sort_values(['sku_id', 'date']).reset_index(drop=True)

# Baseline 1: Naive — tomorrow's demand = today's demand (lag-1)
est['baseline_naive'] = est.groupby('sku_id')['units_sold'].shift(1)

# Baseline 2: Rolling mean — trailing 7-day average (shifted so no leakage: only past data)
est['baseline_rolling7'] = (
    est.groupby('sku_id')['units_sold']
       .transform(lambda s: s.shift(1).rolling(window=7, min_periods=3).mean())
)

# Baseline 3: Weekday-aware — expanding mean of demand on the same weekday, using only past occurrences
est['weekday'] = est['date'].dt.dayofweek
est['baseline_weekday'] = (
    est.groupby(['sku_id', 'weekday'])['units_sold']
       .transform(lambda s: s.shift(1).expanding(min_periods=1).mean())
)

print(est[['sku_id','date','weekday','units_sold','baseline_naive','baseline_rolling7','baseline_weekday']].head(15))
print("\nNulls introduced (expected at series start):")
print(est[['baseline_naive','baseline_rolling7','baseline_weekday']].isna().sum())

      sku_id       date  weekday  units_sold  baseline_naive  \
0   SKU-1000 2026-01-01        3       110.0             NaN   
1   SKU-1000 2026-01-02        4       278.0           110.0   
2   SKU-1000 2026-01-03        5       121.0           278.0   
3   SKU-1000 2026-01-04        6        75.0           121.0   
4   SKU-1000 2026-01-05        0        71.0            75.0   
5   SKU-1000 2026-01-06        1       103.0            71.0   
6   SKU-1000 2026-01-07        2       133.0           103.0   
7   SKU-1000 2026-01-08        3       106.0           133.0   
8   SKU-1000 2026-01-09        4       112.0           106.0   
9   SKU-1000 2026-01-10        5       142.0           112.0   
10  SKU-1000 2026-01-11        6        14.0           142.0   
11  SKU-1000 2026-01-12        0        71.0            14.0   
12  SKU-1000 2026-01-13        1       132.0            71.0   
13  SKU-1000 2026-01-14        2       119.0           132.0   
14  SKU-1000 2026-01-15        3       1

## 5. Chronological holdout: used last 30 days as test, and the rest as "train" (baselines don't need fitting, but we evaluate only on the holdout period to keep this consistent with time-aware validation)

In [7]:
# Chronological holdout: last 30 days as test, rest as "train" (baselines don't need fitting,
# but we evaluate only on the holdout period to keep this consistent with time-aware validation)
cutoff_date = est['date'].max() - pd.Timedelta(days=30)
holdout = est[est['date'] > cutoff_date].copy()

print(f"Holdout period: {holdout['date'].min().date()} to {holdout['date'].max().date()}, {len(holdout)} rows")

def mae(y_true, y_pred):
    mask = y_true.notna() & y_pred.notna()
    return (y_true[mask] - y_pred[mask]).abs().mean(), mask.sum()

def wape(y_true, y_pred):
    mask = y_true.notna() & y_pred.notna()
    return (y_true[mask] - y_pred[mask]).abs().sum() / y_true[mask].sum(), mask.sum()

print("\n--- MAE (units) ---")
for col in ['baseline_naive', 'baseline_rolling7', 'baseline_weekday']:
    m, n = mae(holdout['units_sold'], holdout[col])
    print(f"{col}: MAE={m:.2f}  (n={n})")

print("\n--- WAPE (%) ---")
for col in ['baseline_naive', 'baseline_rolling7', 'baseline_weekday']:
    w, n = wape(holdout['units_sold'], holdout[col])
    print(f"{col}: WAPE={w*100:.2f}%  (n={n})")

Holdout period: 2026-05-31 to 2026-06-29, 750 rows

--- MAE (units) ---
baseline_naive: MAE=75.12  (n=722)
baseline_rolling7: MAE=63.26  (n=736)
baseline_weekday: MAE=52.84  (n=736)

--- WAPE (%) ---
baseline_naive: WAPE=25.23%  (n=722)
baseline_rolling7: WAPE=21.25%  (n=736)
baseline_weekday: WAPE=17.75%  (n=736)


## 6. Hybrid approach (a trial): rolling mean of the same weekday over the last 4 occurrences (approximately 4 weeks), as it probably adapts to the recent trend.

In [8]:
# Hybrid: rolling mean of the same weekday, over the last 4 occurrences (~4 weeks) — adapts to recent trend
est['baseline_weekday_rolling4'] = (
    est.groupby(['sku_id', 'weekday'])['units_sold']
       .transform(lambda s: s.shift(1).rolling(window=4, min_periods=2).mean())
)

holdout = est[est['date'] > cutoff_date].copy()

m, n = mae(holdout['units_sold'], holdout['baseline_weekday_rolling4'])
w, n2 = wape(holdout['units_sold'], holdout['baseline_weekday_rolling4'])
print(f"baseline_weekday_rolling4: MAE={m:.2f}  WAPE={w*100:.2f}%  (n={n})")

print("\n--- Full comparison ---")
for col in ['baseline_naive', 'baseline_rolling7', 'baseline_weekday', 'baseline_weekday_rolling4']:
    m, n = mae(holdout['units_sold'], holdout[col])
    w, _ = wape(holdout['units_sold'], holdout[col])
    print(f"{col}: MAE={m:.2f}  WAPE={w*100:.2f}%  (n={n})")

baseline_weekday_rolling4: MAE=57.19  WAPE=19.21%  (n=736)

--- Full comparison ---
baseline_naive: MAE=75.12  WAPE=25.23%  (n=722)
baseline_rolling7: MAE=63.26  WAPE=21.25%  (n=736)
baseline_weekday: MAE=52.84  WAPE=17.75%  (n=736)
baseline_weekday_rolling4: MAE=57.19  WAPE=19.21%  (n=736)


## 7. Created Lag features

In [9]:
feat = est.copy()

# Lag features (past demand)
for lag in [1, 7, 14]:
    feat[f'lag_{lag}'] = feat.groupby('sku_id')['units_sold'].shift(lag)

# Rolling stats (shifted, so only past data)
feat['rolling_mean_7'] = feat.groupby('sku_id')['units_sold'].transform(lambda s: s.shift(1).rolling(7, min_periods=3).mean())
feat['rolling_mean_14'] = feat.groupby('sku_id')['units_sold'].transform(lambda s: s.shift(1).rolling(14, min_periods=5).mean())
feat['rolling_std_7'] = feat.groupby('sku_id')['units_sold'].transform(lambda s: s.shift(1).rolling(7, min_periods=3).std())

# Weekday-expanding mean (our best baseline) as a feature itself
feat['weekday_expanding_mean'] = feat['baseline_weekday']

# Recent receipts (was anything restocked recently?)
feat['recent_receipt_7'] = feat.groupby('sku_id')['units_received'].transform(lambda s: s.shift(1).rolling(7, min_periods=1).sum())

# Stock position at decision time (yesterday's closing stock — known before today)
feat['prev_closing_stock'] = feat.groupby('sku_id')['closing_stock'].shift(1)

# Calendar
feat['month'] = feat['date'].dt.month
feat['day_of_month'] = feat['date'].dt.day

# Categorical encodings
feat['category_code'] = feat['category'].astype('category').cat.codes
feat['sku_code'] = feat['sku_id'].astype('category').cat.codes

feature_cols = ['lag_1','lag_7','lag_14','rolling_mean_7','rolling_mean_14','rolling_std_7',
                 'weekday_expanding_mean','recent_receipt_7','prev_closing_stock',
                 'weekday','month','day_of_month','category_code','sku_code','lead_time_days']

print("Feature columns:", feature_cols)
print("\nShape before dropping NaN rows:", feat.shape)
feat_model = feat.dropna(subset=feature_cols + ['units_sold']).copy()
print("Shape after dropping rows with any missing feature/target:", feat_model.shape)
print("\nSample:")
print(feat_model[['sku_id','date'] + feature_cols + ['units_sold']].head())

Feature columns: ['lag_1', 'lag_7', 'lag_14', 'rolling_mean_7', 'rolling_mean_14', 'rolling_std_7', 'weekday_expanding_mean', 'recent_receipt_7', 'prev_closing_stock', 'weekday', 'month', 'day_of_month', 'category_code', 'sku_code', 'lead_time_days']

Shape before dropping NaN rows: (4500, 26)
Shape after dropping rows with any missing feature/target: (3793, 26)

Sample:
      sku_id       date  lag_1  lag_7  lag_14  rolling_mean_7  \
14  SKU-1000 2026-01-15  119.0  106.0   110.0       99.428571   
15  SKU-1000 2026-01-16  143.0  112.0   278.0      104.714286   
16  SKU-1000 2026-01-17  137.0  142.0   121.0      108.285714   
17  SKU-1000 2026-01-18   78.0   14.0    75.0       99.142857   
18  SKU-1000 2026-01-19  122.0   71.0    71.0      114.571429   

    rolling_mean_14  rolling_std_7  weekday_expanding_mean  recent_receipt_7  \
14       113.357143      43.900862                   108.0            1657.0   
15       115.714286      46.945764                   195.0            1657.

## 8. Train gradient-boosted model

In [10]:
from sklearn.ensemble import HistGradientBoostingRegressor

train = feat_model[feat_model['date'] <= cutoff_date]
test = feat_model[feat_model['date'] > cutoff_date]

print(f"Train: {train.shape}, Test: {test.shape}")
print(f"Train period: {train['date'].min().date()} to {train['date'].max().date()}")
print(f"Test period: {test['date'].min().date()} to {test['date'].max().date()}")

X_train, y_train = train[feature_cols], train['units_sold']
X_test, y_test = test[feature_cols], test['units_sold']

model = HistGradientBoostingRegressor(max_iter=200, max_depth=4, learning_rate=0.05, random_state=42)
model.fit(X_train, y_train)
pred_gbm = model.predict(X_test)

gbm_mae = (y_test - pred_gbm).abs().mean()
gbm_wape = (y_test - pred_gbm).abs().sum() / y_test.sum()
print(f"\nGBM: MAE={gbm_mae:.2f}  WAPE={gbm_wape*100:.2f}%")

# Fair comparison: baseline_weekday on this SAME test subset (not the full holdout used earlier)
baseline_on_test = test['baseline_weekday']
base_mae = (y_test - baseline_on_test).abs().mean()
base_wape = (y_test - baseline_on_test).abs().sum() / y_test.sum()
print(f"Baseline (weekday) on same subset: MAE={base_mae:.2f}  WAPE={base_wape*100:.2f}%")

print(f"\nImprovement: {(base_wape - gbm_wape)*100:.2f} percentage points")

# Feature importance
importances = pd.Series(model.feature_importances_ if hasattr(model, 'feature_importances_') else None)

Train: (3112, 26), Test: (681, 26)
Train period: 2026-01-15 to 2026-05-30
Test period: 2026-05-31 to 2026-06-29

GBM: MAE=55.69  WAPE=18.74%
Baseline (weekday) on same subset: MAE=53.60  WAPE=18.04%

Improvement: -0.70 percentage points


In [11]:
# SKU-level base rate and volatility (expanding, shifted — only past data)
feat['sku_avg_demand'] = feat.groupby('sku_id')['units_sold'].transform(lambda s: s.shift(1).expanding(min_periods=5).mean())
feat['sku_demand_std'] = feat.groupby('sku_id')['units_sold'].transform(lambda s: s.shift(1).expanding(min_periods=5).std())

# Category-wide rolling demand (cross-SKU signal — captures category-level trend/seasonality)
cat_daily = feat.groupby(['category', 'date'])['units_sold'].sum().reset_index().rename(columns={'units_sold': 'category_total'})
cat_daily = cat_daily.sort_values(['category', 'date'])
cat_daily['category_rolling_mean_7'] = cat_daily.groupby('category')['category_total'].transform(lambda s: s.shift(1).rolling(7, min_periods=3).mean())
feat = feat.merge(cat_daily[['category', 'date', 'category_rolling_mean_7']], on=['category', 'date'], how='left')

# Recent momentum: how far is yesterday's value from the recent average?
feat['recent_deviation'] = feat['lag_1'] - feat['rolling_mean_7']

feature_cols_v2 = feature_cols + ['sku_avg_demand', 'sku_demand_std', 'category_rolling_mean_7', 'recent_deviation']
print("New feature set:", feature_cols_v2)

feat_model_v2 = feat.dropna(subset=feature_cols_v2 + ['units_sold']).copy()
print("Shape:", feat_model_v2.shape)

train2 = feat_model_v2[feat_model_v2['date'] <= cutoff_date]
test2 = feat_model_v2[feat_model_v2['date'] > cutoff_date]
print(f"Train: {train2.shape}, Test: {test2.shape}")

X_train2, y_train2 = train2[feature_cols_v2], train2['units_sold']
X_test2, y_test2 = test2[feature_cols_v2], test2['units_sold']

# Regularized: shallower, fewer iterations, L2 penalty
model2 = HistGradientBoostingRegressor(max_iter=100, max_depth=3, learning_rate=0.05,
                                         l2_regularization=1.0, random_state=42)
model2.fit(X_train2, y_train2)
pred_gbm2 = model2.predict(X_test2)

gbm2_mae = (y_test2 - pred_gbm2).abs().mean()
gbm2_wape = (y_test2 - pred_gbm2).abs().sum() / y_test2.sum()
print(f"\nGBM v2 (enriched features + regularized): MAE={gbm2_mae:.2f}  WAPE={gbm2_wape*100:.2f}%")

baseline_on_test2 = test2['baseline_weekday']
base2_mae = (y_test2 - baseline_on_test2).abs().mean()
base2_wape = (y_test2 - baseline_on_test2).abs().sum() / y_test2.sum()
print(f"Baseline (weekday) on same subset: MAE={base2_mae:.2f}  WAPE={base2_wape*100:.2f}%")
print(f"\nImprovement: {(base2_wape - gbm2_wape)*100:.2f} percentage points")

New feature set: ['lag_1', 'lag_7', 'lag_14', 'rolling_mean_7', 'rolling_mean_14', 'rolling_std_7', 'weekday_expanding_mean', 'recent_receipt_7', 'prev_closing_stock', 'weekday', 'month', 'day_of_month', 'category_code', 'sku_code', 'lead_time_days', 'sku_avg_demand', 'sku_demand_std', 'category_rolling_mean_7', 'recent_deviation']
Shape: (3793, 30)
Train: (3112, 30), Test: (681, 30)

GBM v2 (enriched features + regularized): MAE=53.89  WAPE=18.14%
Baseline (weekday) on same subset: MAE=53.60  WAPE=18.04%

Improvement: -0.10 percentage points


In [12]:
# "Today" = the last date in the dataset; we're scoring risk as of right now, looking forward
today = est['date'].max()
print(f"Decision date (today): {today.date()}")

# Per-SKU, per-weekday average demand using FULL history (this is what the baseline_weekday
# feature approximates row-by-row, but for a live forecast we use the complete history up to today)
weekday_avg = (
    est.groupby(['sku_id', 'weekday'])['units_sold']
       .mean()
       .reset_index()
       .rename(columns={'units_sold': 'weekday_avg_demand'})
)

# Latest known state per SKU: current stock, lead time, category
latest_state = est.sort_values('date').groupby('sku_id').last()[['category', 'closing_stock', 'lead_time_days']].reset_index()
latest_state = latest_state.rename(columns={'closing_stock': 'current_stock'})

# Build horizon forecast: for each SKU, sum expected demand over the next lead_time_days calendar days
horizon_rows = []
for _, row in latest_state.iterrows():
    sku = row['sku_id']
    horizon = int(row['lead_time_days'])
    future_dates = pd.date_range(today + pd.Timedelta(days=1), periods=horizon)
    future_weekdays = future_dates.dayofweek
    sku_wd = weekday_avg[weekday_avg['sku_id'] == sku].set_index('weekday')['weekday_avg_demand']
    forecast_demand = sum(sku_wd.get(wd, sku_wd.mean()) for wd in future_weekdays)
    horizon_rows.append({'sku_id': sku, 'horizon_days': horizon, 'forecast_demand_horizon': forecast_demand})

horizon_df = pd.DataFrame(horizon_rows)
risk_df = latest_state.merge(horizon_df, on='sku_id')
risk_df['shortfall'] = risk_df['forecast_demand_horizon'] - risk_df['current_stock']

print(risk_df.sort_values('shortfall', ascending=False).head(10))

Decision date (today): 2026-06-29
      sku_id category  current_stock  lead_time_days  horizon_days  \
14  SKU-1014   Snacks         1120.0               7             7   
0   SKU-1000   Snacks            0.0               7             7   
18  SKU-1018    Flour         3476.0              14            14   
13  SKU-1013    Pasta         4395.0              10            10   
10  SKU-1010   Snacks          112.0               3             3   
22  SKU-1022   Snacks          235.0               3             3   
23  SKU-1023    Flour         1325.0               7             7   
8   SKU-1008    Sugar          292.0               5             5   
2   SKU-1002    Sugar          659.0               3             3   
20  SKU-1020    Pasta         5757.0              14            14   

    forecast_demand_horizon   shortfall  
14              1891.193077  771.193077  
0                763.372051  763.372051  
18              4115.793846  639.793846  
13              4937.710769

In [13]:
# Stock coverage ratio: how many "horizons" of stock do we currently hold?
risk_df['coverage_ratio'] = risk_df['current_stock'] / risk_df['forecast_demand_horizon']

def risk_tier(ratio):
    if ratio < 0.7:
        return 'High'
    elif ratio < 1.0:
        return 'Medium'
    elif ratio <= 3.0:
        return 'Low'
    else:
        return 'Overstock'

risk_df['risk_tier'] = risk_df['coverage_ratio'].apply(risk_tier)

# Confidence: based on data-quality signal per SKU (flow mismatches in that SKU's history)
flow_check = df_sorted.dropna(subset=['prev_closing','units_received','units_sold','closing_stock'])
flow_check['mismatch'] = flow_check['flow_diff'].abs() >= 0.5
mismatch_rate = flow_check.groupby('sku_id')['mismatch'].mean().reset_index().rename(columns={'mismatch':'flow_mismatch_rate'})

risk_df = risk_df.merge(mismatch_rate, on='sku_id', how='left')
risk_df['flow_mismatch_rate'] = risk_df['flow_mismatch_rate'].fillna(0)

def confidence(row):
    if row['flow_mismatch_rate'] > 0.15:  # >15% of history has flow inconsistencies
        return 'Medium'
    return 'High'

risk_df['confidence'] = risk_df.apply(confidence, axis=1)

print(risk_df.sort_values('coverage_ratio')[['sku_id','category','current_stock','forecast_demand_horizon',
      'coverage_ratio','risk_tier','flow_mismatch_rate','confidence']].to_string(index=False))

print("\nRisk tier distribution:")
print(risk_df['risk_tier'].value_counts())

  sku_id  category  current_stock  forecast_demand_horizon  coverage_ratio risk_tier  flow_mismatch_rate confidence
SKU-1000    Snacks            0.0               763.372051        0.000000      High            0.035714       High
SKU-1010    Snacks          112.0               654.048077        0.171241      High            0.210227     Medium
SKU-1022    Snacks          235.0               594.960000        0.394985      High            0.189655     Medium
SKU-1014    Snacks         1120.0              1891.193077        0.592219      High            0.052632       High
SKU-1008     Sugar          292.0               436.110769        0.669555      High            0.124260       High
SKU-1023     Flour         1325.0              1620.115128        0.817843    Medium            0.000000       High
SKU-1002     Sugar          659.0               801.478205        0.822231    Medium            0.255814     Medium
SKU-1018     Flour         3476.0              4115.793846        0.8445

/tmp/ipykernel_58/149478476.py:18: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  flow_check['mismatch'] = flow_check['flow_diff'].abs() >= 0.5


In [14]:
new_sku_state = df_clean[df_clean['is_new_sku']].sort_values('date').groupby('sku_id').last()[['category', 'closing_stock']].reset_index()
new_sku_state = new_sku_state.rename(columns={'closing_stock': 'current_stock'})

# Most recent lead_time_days is unreliable (varies day-to-day for these SKUs) — still record it, but flag it
new_sku_last_lt = df_clean[df_clean['is_new_sku']].sort_values('date').groupby('sku_id').last()['lead_time_days']
new_sku_state['lead_time_days_last_seen'] = new_sku_state['sku_id'].map(new_sku_last_lt)

# Category-level average daily demand PER SKU (from established SKUs only) as the cold-start demand proxy
category_avg_per_sku_daily = est.groupby('category')['units_sold'].mean()
print("Category average daily demand per SKU (established SKUs):")
print(category_avg_per_sku_daily)

new_sku_state['category_avg_daily_demand'] = new_sku_state['category'].map(category_avg_per_sku_daily)
new_sku_state['forecast_demand_horizon'] = new_sku_state['category_avg_daily_demand'] * new_sku_state['lead_time_days_last_seen']
new_sku_state['coverage_ratio'] = new_sku_state['current_stock'] / new_sku_state['forecast_demand_horizon']
new_sku_state['risk_tier'] = new_sku_state['coverage_ratio'].apply(risk_tier)
new_sku_state['confidence'] = 'Low (cold-start: <2 weeks history, no replenishment observed, unstable lead-time data)'

print("\nNew SKU risk assessment (cold-start path):")
print(new_sku_state[['sku_id','category','current_stock','forecast_demand_horizon','coverage_ratio','risk_tier','confidence']].to_string(index=False))

Category average daily demand per SKU (established SKUs):
category
Beverages    325.219156
Flour        255.634652
Pasta        339.688826
Snacks       257.460302
Sugar        298.413832
Name: units_sold, dtype: float64

New SKU risk assessment (cold-start path):
  sku_id  category  current_stock  forecast_demand_horizon  coverage_ratio risk_tier                                                                             confidence
SKU-2000    Snacks            0.0              2574.603025             0.0      High Low (cold-start: <2 weeks history, no replenishment observed, unstable lead-time data)
SKU-2001 Beverages            0.0              4553.068182             0.0      High Low (cold-start: <2 weeks history, no replenishment observed, unstable lead-time data)
SKU-2002     Flour            0.0              1789.442561             0.0      High Low (cold-start: <2 weeks history, no replenishment observed, unstable lead-time data)


In [15]:
# Unify established + new SKU risk tables into one master table
established_final = risk_df[['sku_id','category','current_stock','lead_time_days','forecast_demand_horizon',
                               'coverage_ratio','risk_tier','confidence','flow_mismatch_rate']].copy()
established_final['is_new_sku'] = False
established_final['confidence_reason'] = established_final['flow_mismatch_rate'].apply(
    lambda r: f"{r*100:.0f}% of history shows inventory-flow inconsistencies" if r > 0.15 else "Clean data history"
)

new_final = new_sku_state[['sku_id','category','current_stock','lead_time_days_last_seen','forecast_demand_horizon',
                             'coverage_ratio','risk_tier','confidence']].copy()
new_final = new_final.rename(columns={'lead_time_days_last_seen':'lead_time_days'})
new_final['is_new_sku'] = True
new_final['flow_mismatch_rate'] = np.nan
new_final['confidence_reason'] = "Cold-start: <2 weeks history, zero replenishment observed, unstable lead-time data"
new_final['confidence'] = 'Low'

master_risk = pd.concat([established_final, new_final], ignore_index=True)
master_risk = master_risk.sort_values(['risk_tier','coverage_ratio'], key=lambda c: c.map({'High':0,'Medium':1,'Low':2,'Overstock':3}) if c.name=='risk_tier' else c)

print(f"Total SKUs scored: {len(master_risk)}")
print("\nRisk tier x Confidence breakdown:")
print(pd.crosstab(master_risk['risk_tier'], master_risk['confidence']))
print("\nFull table:")
print(master_risk[['sku_id','category','risk_tier','confidence','coverage_ratio','current_stock','forecast_demand_horizon']].to_string(index=False))

Total SKUs scored: 28

Risk tier x Confidence breakdown:
confidence  High  Low  Medium
risk_tier                    
High           3    3       2
Low           14    0       1
Medium         4    0       1

Full table:
  sku_id  category risk_tier confidence  coverage_ratio  current_stock  forecast_demand_horizon
SKU-1000    Snacks      High       High        0.000000            0.0               763.372051
SKU-2000    Snacks      High        Low        0.000000            0.0              2574.603025
SKU-2001 Beverages      High        Low        0.000000            0.0              4553.068182
SKU-2002     Flour      High        Low        0.000000            0.0              1789.442561
SKU-1010    Snacks      High     Medium        0.171241          112.0               654.048077
SKU-1022    Snacks      High     Medium        0.394985          235.0               594.960000
SKU-1014    Snacks      High       High        0.592219         1120.0              1891.193077
SKU-1008    

In [16]:
# Recent trend: last 30 days avg demand vs prior 30 days avg demand, per established SKU
last_date = est['date'].max()
recent_30 = est[est['date'] > last_date - pd.Timedelta(days=30)]
prior_30 = est[(est['date'] <= last_date - pd.Timedelta(days=30)) & (est['date'] > last_date - pd.Timedelta(days=60))]

recent_avg = recent_30.groupby('sku_id')['units_sold'].mean().rename('recent_30d_avg')
prior_avg = prior_30.groupby('sku_id')['units_sold'].mean().rename('prior_30d_avg')
trend = pd.concat([recent_avg, prior_avg], axis=1)
trend['trend_pct'] = (trend['recent_30d_avg'] - trend['prior_30d_avg']) / trend['prior_30d_avg'] * 100

# Coefficient of variation (volatility) over full history
cv_stats = est.groupby('sku_id')['units_sold'].agg(['mean','std'])
cv_stats['cv'] = cv_stats['std'] / cv_stats['mean']

# Historical stockout count (established SKUs, correct definition: closing_stock==0, missing excluded)
stockout_hist = est[est['closing_stock']==0].groupby('sku_id').size().rename('historical_stockout_days')

# Recent receipts (last 30 days)
recent_receipts_30 = recent_30.groupby('sku_id')['units_received'].sum().rename('recent_receipts_30d')

# ABC tier (established SKUs only, by total demand)
sku_total_demand = est.groupby('sku_id')['units_sold'].sum().sort_values(ascending=False)
cum_pct = sku_total_demand.cumsum() / sku_total_demand.sum() * 100
abc_tier = cum_pct.apply(lambda p: 'A' if p <= 80 else ('B' if p <= 95 else 'C')).rename('abc_tier')

evidence_supplement = pd.concat([trend[['trend_pct']], cv_stats[['cv']], stockout_hist, recent_receipts_30, abc_tier], axis=1).reset_index()
evidence_supplement = evidence_supplement.rename(columns={'index':'sku_id'})
evidence_supplement['historical_stockout_days'] = evidence_supplement['historical_stockout_days'].fillna(0)
evidence_supplement['recent_receipts_30d'] = evidence_supplement['recent_receipts_30d'].fillna(0)

print(evidence_supplement.to_string(index=False))

  sku_id  trend_pct       cv  historical_stockout_days  recent_receipts_30d abc_tier
SKU-1000  -4.000997 0.291738                       6.0                 1824        C
SKU-1001   7.113435 0.192712                       7.0                12176        A
SKU-1002   4.072988 0.242623                      45.0                 7492        B
SKU-1003  -0.191050 0.294744                       1.0                 5340        B
SKU-1004  -0.650759 0.251626                       8.0                 6166        C
SKU-1005   1.773424 0.206732                       3.0                 3839        C
SKU-1006  -8.318188 0.277514                       0.0                18775        A
SKU-1007  -2.513591 0.342555                       2.0                16164        A
SKU-1008   4.825422 0.202515                      22.0                 2530        C
SKU-1009   2.449461 0.290797                       0.0                12757        A
SKU-1010  -2.034514 0.211337                      38.0           

In [17]:
master_risk_full = master_risk.merge(evidence_supplement, on='sku_id', how='left')

def build_evidence(sku_row):
    """Structured evidence object — the ONLY thing the LLM sees. No raw data access, no calculation."""
    ev = {
        "sku_id": sku_row['sku_id'],
        "category": sku_row['category'],
        "risk_tier": sku_row['risk_tier'],
        "confidence": sku_row['confidence'],
        "current_stock_units": round(sku_row['current_stock'], 0),
        "lead_time_days": int(sku_row['lead_time_days']),
        "forecasted_demand_over_lead_time": round(sku_row['forecast_demand_horizon'], 0),
        "shortfall_or_surplus_units": round(sku_row['forecast_demand_horizon'] - sku_row['current_stock'], 0),
        "coverage_ratio": round(sku_row['coverage_ratio'], 2),
        "is_new_sku": bool(sku_row['is_new_sku']),
    }
    if sku_row['is_new_sku']:
        ev["confidence_reason"] = sku_row['confidence_reason']
        ev["note"] = "New SKU: demand forecast uses category-average as a proxy, not this SKU's own history."
    else:
        ev["demand_trend_last_30d_pct"] = round(sku_row['trend_pct'], 1) if pd.notna(sku_row['trend_pct']) else None
        ev["demand_volatility_cv"] = round(sku_row['cv'], 2) if pd.notna(sku_row['cv']) else None
        ev["historical_stockout_days_last_6mo"] = int(sku_row['historical_stockout_days'])
        ev["recent_receipts_last_30d"] = int(sku_row['recent_receipts_30d'])
        ev["abc_tier"] = sku_row['abc_tier']
        ev["confidence_reason"] = sku_row['confidence_reason']
    return ev

# Test on a couple of flagged (High risk) SKUs
high_risk_examples = master_risk_full[master_risk_full['risk_tier']=='High'].head(3)
for _, row in high_risk_examples.iterrows():
    print(build_evidence(row))
    print()

{'sku_id': 'SKU-1000', 'category': 'Snacks', 'risk_tier': 'High', 'confidence': 'High', 'current_stock_units': 0.0, 'lead_time_days': 7, 'forecasted_demand_over_lead_time': 763.0, 'shortfall_or_surplus_units': 763.0, 'coverage_ratio': 0.0, 'is_new_sku': False, 'demand_trend_last_30d_pct': -4.0, 'demand_volatility_cv': 0.29, 'historical_stockout_days_last_6mo': 6, 'recent_receipts_last_30d': 1824, 'abc_tier': 'C', 'confidence_reason': 'Clean data history'}

{'sku_id': 'SKU-2000', 'category': 'Snacks', 'risk_tier': 'High', 'confidence': 'Low', 'current_stock_units': 0.0, 'lead_time_days': 10, 'forecasted_demand_over_lead_time': 2575.0, 'shortfall_or_surplus_units': 2575.0, 'coverage_ratio': 0.0, 'is_new_sku': True, 'confidence_reason': 'Cold-start: <2 weeks history, zero replenishment observed, unstable lead-time data', 'note': "New SKU: demand forecast uses category-average as a proxy, not this SKU's own history."}

{'sku_id': 'SKU-2001', 'category': 'Beverages', 'risk_tier': 'High', 'c

In [18]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
DEEPSEEK_API_KEY = user_secrets.get_secret("DEEPSEEK_API_KEY")
print("Key loaded:", DEEPSEEK_API_KEY[:6] + "..." if DEEPSEEK_API_KEY else "MISSING")

Key loaded: sk-78d...


In [19]:
!pip install -q openai

import os
os.environ["DEEPSEEK_API_KEY"] = DEEPSEEK_API_KEY

from openai import OpenAI

client = OpenAI(api_key=DEEPSEEK_API_KEY, base_url="https://api.deepseek.com")

# Quick connectivity test
response = client.chat.completions.create(
    model="deepseek-chat",
    messages=[{"role": "user", "content": "Reply with exactly: connection ok"}],
    max_tokens=10,
)
print(response.choices[0].message.content)

connection ok


In [20]:
SYSTEM_PROMPT = """You are a supply-chain analyst assistant for Flour Mills of Nigeria.
You explain inventory risk flags to operations staff in plain English.

Rules you MUST follow:
- Use ONLY the numbers given to you in the evidence. Never invent, estimate, or
  round numbers you were not given.
- Never change or second-guess the risk_tier or confidence value provided —
  those were already decided by the analytical system. Your job is only to
  explain WHY, using the evidence.
- Be concise: 2-4 sentences.
- If confidence is Low, say so plainly and explain why (e.g. new SKU / data quality),
  so the reader knows to treat the number with appropriate caution.
- Do not make causal claims the evidence doesn't support.
"""

def explain_sku(evidence, client):
    user_prompt = f"Explain this SKU's inventory risk flag using only this evidence:\n\n{evidence}"
    response = client.chat.completions.create(
        model="deepseek-chat",
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt},
        ],
        temperature=0.3,
        max_tokens=300,
    )
    return response.choices[0].message.content.strip()

# Test on the two examples we built earlier: SKU-1000 (established, High risk) and SKU-2000 (new, cold-start)
example_established = build_evidence(master_risk_full[master_risk_full['sku_id']=='SKU-1000'].iloc[0])
example_new = build_evidence(master_risk_full[master_risk_full['sku_id']=='SKU-2000'].iloc[0])

print("=== SKU-1000 (established) ===")
print("Evidence:", example_established)
print("\nExplanation:")
print(explain_sku(example_established, client))

print("\n\n=== SKU-2000 (new/cold-start) ===")
print("Evidence:", example_new)
print("\nExplanation:")
print(explain_sku(example_new, client))

=== SKU-1000 (established) ===
Evidence: {'sku_id': 'SKU-1000', 'category': 'Snacks', 'risk_tier': 'High', 'confidence': 'High', 'current_stock_units': np.float64(0.0), 'lead_time_days': 7, 'forecasted_demand_over_lead_time': np.float64(763.0), 'shortfall_or_surplus_units': np.float64(763.0), 'coverage_ratio': np.float64(0.0), 'is_new_sku': False, 'demand_trend_last_30d_pct': np.float64(-4.0), 'demand_volatility_cv': np.float64(0.29), 'historical_stockout_days_last_6mo': 6, 'recent_receipts_last_30d': 1824, 'abc_tier': 'C', 'confidence_reason': 'Clean data history'}

Explanation:
SKU-1000 is flagged as High risk because current stock is 0 units while forecasted demand over the 7-day lead time is 763 units, leaving a shortfall of 763 units and a coverage ratio of 0.0. This means you have no stock to cover any of the expected demand during the lead time. The confidence in this flag is High, based on clean data history, so you can rely on this assessment.


=== SKU-2000 (new/cold-start) =

In [21]:
def get_highest_risk_skus(n=5):
    return master_risk_full[master_risk_full['risk_tier']=='High'].sort_values('coverage_ratio')[
        ['sku_id','category','risk_tier','confidence','coverage_ratio','shortfall' if 'shortfall' in master_risk_full.columns else 'current_stock']
    ].head(n).to_dict('records')

def get_sku_detail(sku_id):
    row = master_risk_full[master_risk_full['sku_id']==sku_id]
    if row.empty:
        return {"error": f"{sku_id} not found"}
    return build_evidence(row.iloc[0])

def get_stockouts_by_category():
    # uses corrected stockout definition
    stockouts = est[stockout_mask(est)] if 'stockout_mask' in dir() else est[est['closing_stock']==0]
    return stockouts.groupby('category').size().sort_values(ascending=False).to_dict()

def get_demand_trend_last_month():
    return evidence_supplement[['sku_id','trend_pct']].sort_values('trend_pct').to_dict('records')

def get_high_demand_high_variability_skus(n=5):
    merged = evidence_supplement.merge(est.groupby('sku_id')['units_sold'].mean().rename('avg_demand'), on='sku_id')
    return merged.sort_values(['avg_demand','cv'], ascending=[False, False])[['sku_id','avg_demand','cv']].head(n).to_dict('records')

# Quick sanity check on each
print("Highest risk:", get_highest_risk_skus(3))
print("\nSKU-1017 detail:", get_sku_detail('SKU-1017'))
print("\nStockouts by category:", get_stockouts_by_category())

Highest risk: [{'sku_id': 'SKU-1000', 'category': 'Snacks', 'risk_tier': 'High', 'confidence': 'High', 'coverage_ratio': 0.0, 'current_stock': 0.0}, {'sku_id': 'SKU-2000', 'category': 'Snacks', 'risk_tier': 'High', 'confidence': 'Low', 'coverage_ratio': 0.0, 'current_stock': 0.0}, {'sku_id': 'SKU-2001', 'category': 'Beverages', 'risk_tier': 'High', 'confidence': 'Low', 'coverage_ratio': 0.0, 'current_stock': 0.0}]

SKU-1017 detail: {'sku_id': 'SKU-1017', 'category': 'Beverages', 'risk_tier': 'Low', 'confidence': 'Medium', 'current_stock_units': np.float64(1691.0), 'lead_time_days': 3, 'forecasted_demand_over_lead_time': np.float64(956.0), 'shortfall_or_surplus_units': np.float64(-735.0), 'coverage_ratio': np.float64(1.77), 'is_new_sku': False, 'demand_trend_last_30d_pct': np.float64(-2.8), 'demand_volatility_cv': np.float64(0.21), 'historical_stockout_days_last_6mo': 45, 'recent_receipts_last_30d': 7276, 'abc_tier': 'A', 'confidence_reason': '25% of history shows inventory-flow inconsi

In [22]:
QA_INTENTS = """You must classify the user's question into exactly one of these intents,
and extract any parameters needed. Respond with ONLY valid JSON, no other text.

Available intents:
- "highest_risk": which SKUs are highest risk. params: {"n": int, default 5}
- "sku_detail": details/explanation for a specific SKU. params: {"sku_id": string, e.g. "SKU-1017"}
- "stockouts_by_category": which category has most stockouts. params: {}
- "demand_trend": how demand has changed recently. params: {}
- "high_demand_high_variability": which high-demand SKUs are also volatile. params: {"n": int, default 5}
- "unknown": question doesn't match any of the above. params: {}

Respond in this exact JSON format: {"intent": "...", "params": {...}}
"""

def classify_question(question, client):
    response = client.chat.completions.create(
        model="deepseek-chat",
        messages=[
            {"role": "system", "content": QA_INTENTS},
            {"role": "user", "content": question},
        ],
        temperature=0,
        max_tokens=100,
    )
    import json
    return json.loads(response.choices[0].message.content.strip())

def answer_question(question, client):
    classification = classify_question(question, client)
    intent, params = classification['intent'], classification.get('params', {})

    if intent == 'highest_risk':
        data = get_highest_risk_skus(params.get('n', 5))
    elif intent == 'sku_detail':
        data = get_sku_detail(params.get('sku_id', ''))
    elif intent == 'stockouts_by_category':
        data = get_stockouts_by_category()
    elif intent == 'demand_trend':
        data = get_demand_trend_last_month()
    elif intent == 'high_demand_high_variability':
        data = get_high_demand_high_variability_skus(params.get('n', 5))
    else:
        return "I can answer questions about SKU risk, stockouts, demand trends, and volatility — could you rephrase?"

    answer_prompt = f"""Answer this question using ONLY the data below. Be concise (2-4 sentences).
Do not invent numbers not present in the data.

Question: {question}

Data: {data}
"""
    response = client.chat.completions.create(
        model="deepseek-chat",
        messages=[{"role": "user", "content": answer_prompt}],
        temperature=0.3,
        max_tokens=250,
    )
    return response.choices[0].message.content.strip()

# Test with the brief's example questions
test_questions = [
    "Which SKUs are at highest risk?",
    "Why is SKU-1017 flagged?",
    "Which category has the most stockouts?",
]
for q in test_questions:
    print(f"Q: {q}")
    print(f"A: {answer_question(q, client)}")
    print()

Q: Which SKUs are at highest risk?
A: Based on the data, SKU-1000 is at the highest risk because it has both a High risk tier and High confidence, with zero current stock and zero coverage. The other SKUs (SKU-2000, SKU-2001, SKU-2002) also have zero stock but are flagged with Low confidence, making their risk assessment less certain. SKU-1010 has some stock (112 units) and a positive coverage ratio, so it is less at risk than the others.

Q: Why is SKU-1017 flagged?
A: SKU-1017 is flagged due to a **shortfall of -735 units** (forecasted demand over lead time is 956 units, while current stock is 1,691 units), despite a **coverage ratio of 1.77** and a **Low risk tier**. The flag is also supported by **Medium confidence** because 25% of its history shows inventory-flow inconsistencies, and it has **45 historical stockout days** in the last 6 months.

Q: Which category has the most stockouts?
A: Snacks has the most stockouts, with 88 reported. This is higher than Sugar (69), Beverages (6